# Projeto Final — Parte 2: Engenharia de Recursos e Modelagem Avançada

## Contexto da Segunda Etapa
Após implementar o pipeline básico com uma Árvore de Decisão simples, a equipe de engenharia da fábrica identificou que o modelo inicial pode ser aprimorado. Em cenários reais de IoT industrial, os sensores interagem entre si, os dados podem sofrer com variações estatísticas na divisão de treino/teste, e modelos simples podem decorar os dados (*overfitting*).

Nesta etapa avançada, você assumirá o papel de um **Cientista de Dados Sênior** para implementar técnicas que aproximam este projeto de uma solução real de produção.

O desafio desta segunda etapa consiste em:
1. **Engenharia de Recursos (Feature Engineering):** Criar novas variáveis combinando sensores para capturar padrões complexos de desgaste;
2. **Validação Cruzada (Cross-Validation):** Garantir uma avaliação estatisticamente robusta do modelo;
3. **Otimização de Hiperparâmetros (Grid Search):** Encontrar o melhor ajuste de parâmetros para evitar que a árvore "decore" os dados;
4. **Modelagem Ensemble (Random Forest):** Testar um algoritmo de floresta de decisão para comparar o desempenho;
5. **Análise de Importância de Recursos (Feature Importance):** Identificar quais sensores e métricas são os mais críticos para prever as falhas.

---

## Objetivos de aprendizagem
Ao final desta etapa, você deverá ser capaz de:

- Criar e testar novas variáveis baseadas no conhecimento de domínio (sensores industriais);
- Aplicar validação cruzada para estimar a capacidade de generalização de um modelo;
- Realizar buscas sistemáticas pelos melhores parâmetros de um algoritmo;
- Avaliar o ganho de performance ao migrar de um modelo simples para um modelo ensemble;
- Interpretar quais variáveis físicas são os principais gatilhos para anomalias no maquinário.

---

## Entrega esperada
No final do notebook, esta seção adicional deve conter:

- Código de criação das novas variáveis e sua justificativa física;
- Teste de validação cruzada com relatório de acurácia média e desvio padrão;
- Relatório dos melhores parâmetros encontrados pelo Grid Search;
- Treinamento e métricas completas do modelo Random Forest;
- Gráfico de barras horizontais ilustrando a importância das variáveis;
- Conclusão final expandida respondendo às novas questões de negócio.

## 1. Importação de Novas Bibliotecas

Para esta etapa avançada, vamos precisar de novas funções do `scikit-learn`. 
Sua tarefa é pesquisar e importar as ferramentas corretas para as seguintes atividades:
1. Realizar **Validação Cruzada** (`cross_val_score`);
2. Realizar busca de hiperparâmetros por grade (**Grid Search**) (`GridSearchCV`);
3. Utilizar o modelo de conjunto **Random Forest** (`RandomForestClassifier`).

In [ ]:
# Imports necessários para a Etapa 2

# Validação Cruzada
from sklearn.model_selection import cross_val_score

# Busca de hiperparâmetros em grade
from sklearn.model_selection import GridSearchCV

# Modelo ensemble Random Forest
from sklearn.ensemble import RandomForestClassifier

print("Se os imports rodaram sem erros, suas importações estão corretas!")


## 2. Engenharia de Recursos (Feature Engineering)

Em ambientes industriais de IoT, os sensores raramente devem ser analisados apenas de forma isolada. Variáveis combinadas podem revelar dinâmicas de falha muito mais complexas.

**Sua Tarefa:**
Crie duas novas colunas em uma cópia do seu dataframe tratado (`df_tratado` da Etapa 1):
1. **`termica_pressao`**: O produto (multiplicação) das colunas de `temperatura` e `pressao`. Ela visa mapear o estresse físico-térmico combinado do equipamento.
2. **`desgaste_acumulado`**: O produto da `vibracao` pelo `tempo_de_operacao`. Ela visa mapear a vibração acumulada ao longo da vida útil do ativo.

In [ ]:
# Primeiro, crie uma cópia do seu DataFrame tratado obtido no projeto anterior
X_avancado = df_tratado.drop("falha", axis=1).copy()
y = df_tratado["falha"]

# Criando as novas variáveis de engenharia de recursos
X_avancado["termica_pressao"] = X_avancado["temperatura"] * X_avancado["pressao"]
X_avancado["desgaste_acumulado"] = X_avancado["vibracao"] * X_avancado["tempo_operacao"]

# Verificando se as novas colunas foram criadas corretamente
print("Colunas atuais no dataset:", X_avancado.columns.tolist())
X_avancado.head()


### Reflexão do Aluno
Pesquise e explique: Por que a criação de novas variáveis (Feature Engineering) baseadas no conhecimento de domínio (no caso, física industrial e mecânica) pode melhorar o poder preditivo de um modelo comparado ao uso de sensores puros?

In [ ]:
# Escreva aqui sua explicação teórica

## 3. Validação Cruzada (Cross-Validation)

Até agora, avaliamos o modelo usando uma divisão única de treino/teste (Holdout). No entanto, essa abordagem pode sofrer com variações estatísticas de amostragem.

**Sua Tarefa:**
Pesquise sobre o funcionamento do **K-Fold Cross-Validation** e utilize a função `cross_val_score` para testar um modelo básico de `DecisionTreeClassifier(random_state=42)` com **5 dobras (folds)** sobre os seus dados avançados (`X_avancado` e `y`).

In [ ]:
# Instanciando o classificador básico de árvore de decisão
modelo_base_cv = DecisionTreeClassifier(random_state=42)

# Validação cruzada com 5 dobras (folds), usando acurácia como métrica
scores = cross_val_score(modelo_base_cv, X_avancado, y, cv=5, scoring="accuracy")

# Exibindo os resultados obtidos de cada dobra
print("Acurácias por fold:", scores)
print(f"Acurácia média obtida: {scores.mean():.2%}")
print(f"Desvio padrão das acurácias: {scores.std():.4f}")


## 4. Otimização de Hiperparâmetros (Grid Search)

Árvores de decisão são propensas ao *overfitting* se crescerem sem restrições de profundidade ou de amostragem nas folhas. Para encontrar a árvore ideal, usamos busca em grade (*Grid Search*) que testa de forma combinada diversos parâmetros limitantes.

**Sua Tarefa:**
Defina o dicionário de parâmetros (`param_grid`) e configure um buscador `GridSearchCV` para testar as melhores combinações de poda da árvore.

In [ ]:
# Dicionário de hiperparâmetros a serem testados
param_grid = {
    "max_depth": [3, 5, 7, None],               # Testar profundidades máximas diferentes
    "min_samples_split": [2, 5, 10],             # Mínimo de amostras para dividir um nó
    "min_samples_leaf": [1, 2, 4],               # Mínimo de amostras permitidas em um nó folha
    "criterion": ["gini", "entropy"]             # Critério de medição de qualidade da divisão
}

# Configurando o GridSearchCV
grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy"
)

# Treinando a busca em grade sobre todo o conjunto de dados avançado
grid_search.fit(X_avancado, y)

# Exibindo os resultados obtidos da melhor combinação
print("Melhores Hiperparâmetros:", grid_search.best_params_)
print(f"Melhor acurácia de validação cruzada obtida: {grid_search.best_score_:.2%}")


## 5. Comparação com Modelos Ensemble: Random Forest

Ensembles combinam previsões de vários classificadores individuais para reduzir a variância e aumentar a robustez. O **Random Forest** cria centenas de árvores de decisão diferentes com amostragem aleatória dos dados e das colunas.

**Sua Tarefa:**
1. Divida o conjunto avançado (`X_avancado`, `y`) em treino (70%) e teste (30%) usando o `train_test_split`;
2. Treine um estimador `RandomForestClassifier(random_state=42)` com o conjunto de treino;
3. Faça previsões para o conjunto de teste e exiba o relatório de classificação correspondente.

In [ ]:
# Divisão dos dados avançados em treino (70%) e teste (30%)
X_train_av, X_test_av, y_train_av, y_test_av = train_test_split(
    X_avancado, y, test_size=0.3, random_state=42
)

# Instanciando o classificador Random Forest com 100 árvores
modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42)

# Treinando o modelo
modelo_rf.fit(X_train_av, y_train_av)

# Gerando as predições para o conjunto de teste
y_pred_rf = modelo_rf.predict(X_test_av)

# Calculando e exibindo a acurácia e o relatório de classificação
acc_rf = accuracy_score(y_test_av, y_pred_rf)
print(f"Acurácia do Random Forest: {acc_rf:.2%}")
print(classification_report(y_test_av, y_pred_rf))


## 6. Importância dos Recursos (Feature Importance)

Uma propriedade muito útil dos algoritmos baseados em árvores é a capacidade de avaliar a contribuição relativa de cada atributo para a pureza das divisões da árvore, revelando quais sensores são fundamentais.

**Sua Tarefa:**
Acesse as importâncias calculadas pelo seu modelo Random Forest, ordene-as e exiba-as em um gráfico de barras horizontal utilizando a biblioteca `matplotlib`.

In [ ]:
# Importância das variáveis calculada pelo Random Forest
importancias = modelo_rf.feature_importances_

# Mapeando os nomes dos recursos
nomes_features = X_avancado.columns

# Criando um DataFrame para ordenação e plotagem
df_importancia = pd.DataFrame({
    "Sensor": nomes_features,
    "Importância": importancias
}).sort_values(by="Importância", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(df_importancia["Sensor"], df_importancia["Importância"])

plt.title("Importância dos Sensores e Recursos no Diagnóstico de Falhas")
plt.xlabel("Grau de Importância (0 a 1)")
plt.ylabel("Variáveis do Dataset")
plt.tight_layout()
plt.show()


## 7. Conclusão da Segunda Etapa

Responda às seguintes perguntas finais sobre a aplicação de metodologias avançadas de Machine Learning:

1. As novas variáveis criadas (`termica_pressao` e `desgaste_acumulado`) agregaram valor preditivo? Explique com base na variação de acurácia obtida no projeto.
2. Qual a utilidade prática da **Validação Cruzada** em um contexto em que o dataset possui poucos registros (ex: apenas 200)?
3. Como o ajuste de hiperparâmetros por **Grid Search** ajuda o time de engenharia a garantir que o modelo não irá falhar ao receber novos dados gerados em tempo real na fábrica?
4. Observando o gráfico de **Importância dos Recursos**, qual variável ou sensor físico é o indicador mais crítico para a detecção precoce de falhas? Se você fosse o engenheiro responsável, em qual sensor focaria o investimento em calibração constante?

In [ ]:
# Escreva aqui suas respostas finais fundamentadas em dados

# Checklist de Entrega da Etapa Avançada

Confirme se você completou todos os desafios:

- [ ] Importações corretas dos submódulos de validação cruzada, grid search e random forest.
- [ ] Equações matemáticas corretas no Pandas para as novas variáveis físicas.
- [ ] Execução da validação cruzada K-Fold com cálculo correto de média e desvio padrão.
- [ ] Configuração de parâmetros e treinamento bem-sucedido do Grid Search.
- [ ] Divisão de treino e teste dedicada para o modelo ensemble e treinamento do Random Forest.
- [ ] Plotagem correta do gráfico de barras horizontais da importância das colunas.
- [ ] Análise conceitual das dinâmicas de engenharia de sensores e validação de dados industriais nas células de conclusão.